# Week 10: Practical Deep Learning Systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/10/Week_10_Practical_Deep_Learning_Systems.ipynb)

**Course:** Neural Architectures and Representation Learning (Master level)

## Learning goals

By the end of this week you should be able to:

- treat a training run as evidence, not as a one-off result;
- log configurations, metrics, histories, and artifacts in a reproducible way;
- compare multiple runs without relying on memory;
- explain the idea behind hyperparameter tuning;
- decide when to train from scratch, freeze a representation, or fine-tune it.

The class habit becomes:

`configure -> run -> track -> compare -> decide -> explain`

### External anchors for this week

These are optional references, not required dependencies. The notebook uses a lightweight in-memory tracker so it stays Colab-friendly.

| Chapter | Resource | Why it helps |
|---|---|---|
| Experiment tracking | [MLflow Tracking docs](https://mlflow.org/docs/latest/ml/tracking/) | Standard vocabulary: experiments, runs, parameters, metrics, and artifacts. |
| Experiment dashboards | [Weights & Biases docs](https://docs.wandb.ai/) | Shows the production version of the run-comparison workflow we build manually. |
| Local-first tracking | [Hugging Face Trackio docs](https://huggingface.co/docs/trackio/index) | Lightweight local dashboard, W&B-style API, and optional Hugging Face Spaces sharing. |
| Hyperparameter tuning | [Optuna docs](https://optuna.readthedocs.io/) | Official reference for define-by-run hyperparameter optimization. |
| Fine-tuning workflow | [Hugging Face fine-tuning guide](https://huggingface.co/docs/transformers/en/training) | Practical bridge from this tiny notebook to modern pretrained-model workflows. |
| MLOps overview | [Google: Rules of ML](https://developers.google.com/machine-learning/guides/rules-of-ml) | Useful engineering heuristics for avoiding premature complexity. |

---

## Environment

This notebook uses `torch`, `numpy`, `matplotlib`, and `scikit-learn`. CPU is enough.

## Colab pointers

Run cells in order. The central objects are:

- `RunTracker`: stores config, metrics, history, and small artifacts.
- `RepresentationMLP`: a small network with a visible representation layer.
- `run_experiment(...)`: trains one model and logs one run.
- `tracker.table()`: turns all runs into a comparison table.

In [ ]:
import copy
import math
import random
import time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

print("torch:", torch.__version__)
print("numpy:", np.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(10)

---

## 1. A small practical task

We use the built-in digits dataset: 8x8 grayscale images of handwritten digits. This keeps the system small enough that run comparison matters more than compute.

The representation-learning question is:

> Which training setup learns a useful hidden representation, and how do we know?

In [ ]:
digits = load_digits()
X = digits.data.astype("float32") / 16.0
y = digits.target.astype("int64")

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=10, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=10, stratify=y_train_full
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

splits = {
    "train": (torch.tensor(X_train), torch.tensor(y_train)),
    "val": (torch.tensor(X_val), torch.tensor(y_val)),
    "test": (torch.tensor(X_test), torch.tensor(y_test)),
}

print("train / val / test:", len(y_train), len(y_val), len(y_test))
print("input dimension:", X_train.shape[1])

fig, axes = plt.subplots(2, 8, figsize=(9, 2.6))
for ax, image, label in zip(axes.ravel(), digits.images[:16], digits.target[:16]):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(str(label))
    ax.axis("off")
plt.suptitle("Example 8x8 digit images")
plt.tight_layout()
plt.show()

---

## 2. A tiny representation model

The model has a visible representation layer:

`input pixels -> hidden layer -> representation -> classifier`

This is deliberately simple. Week 10 is about the workflow around training, not about inventing a new architecture.

In [ ]:
class RepresentationMLP(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=64, rep_dim=16, output_dim=10, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, rep_dim),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(rep_dim, output_dim)

    def forward(self, x, return_representation=False):
        rep = self.features(x)
        logits = self.classifier(rep)
        if return_representation:
            return logits, rep
        return logits


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def batch_iter(X, y, batch_size=128, shuffle=True, seed=0):
    n = len(y)
    indices = np.arange(n)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)
    for start in range(0, n, batch_size):
        idx = indices[start:start + batch_size]
        yield X[idx].to(device), y[idx].to(device)


@torch.no_grad()
def evaluate(model, X, y, batch_size=256):
    model.eval()
    losses, correct, total = [], 0, 0
    all_preds = []
    for xb, yb in batch_iter(X, y, batch_size=batch_size, shuffle=False):
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        pred = logits.argmax(dim=1)
        losses.append(loss.item() * len(yb))
        correct += int((pred == yb).sum().item())
        total += len(yb)
        all_preds.append(pred.cpu())
    return {
        "loss": float(sum(losses) / total),
        "accuracy": float(correct / total),
        "predictions": torch.cat(all_preds),
    }


def train_model(model, train_split, val_split, epochs=25, lr=0.01, weight_decay=0.0, batch_size=128, seed=10):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    Xtr, ytr = train_split
    Xv, yv = val_split
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_state, best_val = None, -1.0

    for epoch in range(epochs):
        model.train()
        for xb, yb in batch_iter(Xtr, ytr, batch_size=batch_size, shuffle=True, seed=seed + epoch):
            optimizer.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()

        train_metrics = evaluate(model, Xtr, ytr)
        val_metrics = evaluate(model, Xv, yv)
        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["accuracy"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])

        if val_metrics["accuracy"] > best_val:
            best_val = val_metrics["accuracy"]
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

---

## 3. A tiny experiment tracker

Real tools like MLflow, W&B, and Trackio give you dashboards, artifacts, model registries, and collaboration features. Underneath, the habit is simple:

- record the config;
- record the metrics;
- keep the history;
- compare runs with the same validation split;
- choose a model for a reason.

In [ ]:
class RunTracker:
    def __init__(self, experiment_name):
        self.experiment_name = experiment_name
        self.runs = []

    def log_run(self, name, config, metrics, history, artifacts=None):
        record = {
            "run_id": f"run_{len(self.runs):03d}",
            "name": name,
            "config": dict(config),
            "metrics": dict(metrics),
            "history": history,
            "artifacts": artifacts or {},
        }
        self.runs.append(record)
        return record

    def table(self):
        rows = []
        for run in self.runs:
            row = {"run_id": run["run_id"], "name": run["name"]}
            row.update(run["config"])
            row.update(run["metrics"])
            rows.append(row)
        return rows

    def best(self, metric="val_accuracy"):
        return max(self.runs, key=lambda run: run["metrics"][metric])


def print_run_table(rows, columns=None):
    if not rows:
        print("No runs yet.")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {col: max(len(col), max(len(f"{row.get(col, '')}") for row in rows)) for col in columns}
    header = " | ".join(col.ljust(widths[col]) for col in columns)
    print(header)
    print("-+-".join("-" * widths[col] for col in columns))
    for row in rows:
        print(" | ".join(f"{row.get(col, '')}".ljust(widths[col]) for col in columns))


def plot_histories(tracker, metric="val_acc", title="Validation accuracy by run"):
    plt.figure(figsize=(8, 4))
    for run in tracker.runs:
        values = run["history"].get(metric, [])
        if values:
            plt.plot(values, label=run["name"])
    plt.xlabel("epoch")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_final_metrics(tracker):
    rows = tracker.table()
    names = [row["name"] for row in rows]
    val = [row["val_accuracy"] for row in rows]
    test = [row["test_accuracy"] for row in rows]
    x = np.arange(len(names))
    plt.figure(figsize=(max(7, len(names) * 0.8), 4))
    plt.bar(x - 0.18, val, width=0.36, label="validation")
    plt.bar(x + 0.18, test, width=0.36, label="test")
    plt.xticks(x, names, rotation=30, ha="right")
    plt.ylim(0.75, 1.01)
    plt.ylabel("accuracy")
    plt.title("Final run comparison")
    plt.legend()
    plt.tight_layout()
    plt.show()


tracker = RunTracker("week10_digits_workflow")

---

## 4. Baseline run

A baseline is not meant to be impressive. It is the reference point that makes later claims meaningful.

In [ ]:
def run_experiment(name, config, tracker=tracker, verbose=True):
    set_seed(config.get("seed", 10))
    start = time.time()
    model = RepresentationMLP(
        input_dim=64,
        hidden_dim=config["hidden_dim"],
        rep_dim=config["rep_dim"],
        output_dim=10,
        dropout=config.get("dropout", 0.0),
    )
    param_count = count_parameters(model)
    model, history = train_model(
        model,
        splits["train"],
        splits["val"],
        epochs=config["epochs"],
        lr=config["lr"],
        weight_decay=config.get("weight_decay", 0.0),
        batch_size=config.get("batch_size", 128),
        seed=config.get("seed", 10),
    )
    test_metrics = evaluate(model, *splits["test"])
    metrics = {
        "best_val_accuracy": round(max(history["val_acc"]), 4),
        "val_accuracy": round(evaluate(model, *splits["val"])["accuracy"], 4),
        "test_accuracy": round(test_metrics["accuracy"], 4),
        "final_train_accuracy": round(history["train_acc"][-1], 4),
        "parameters": param_count,
        "seconds": round(time.time() - start, 2),
    }
    record = tracker.log_run(
        name=name,
        config=config,
        metrics=metrics,
        history=history,
        artifacts={"model": model, "test_predictions": test_metrics["predictions"]},
    )
    if verbose:
        print(name, metrics)
    return model, record

baseline_config = {
    "hidden_dim": 48,
    "rep_dim": 12,
    "dropout": 0.0,
    "lr": 0.01,
    "weight_decay": 0.0,
    "batch_size": 128,
    "epochs": 24,
    "seed": 10,
}

baseline_model, baseline_run = run_experiment("baseline", baseline_config)
print_run_table(tracker.table(), ["run_id", "name", "hidden_dim", "rep_dim", "lr", "dropout", "val_accuracy", "test_accuracy", "parameters", "seconds"])
plot_histories(tracker, "val_acc", "Baseline validation curve")

### Baseline reflection

Before changing anything, ask:

1. What is the validation score?
2. Is the training score much higher than validation?
3. How many parameters did this model use?
4. What would be a controlled next change?

---

## 5. Compare several controlled runs

A run comparison is only useful when the changes are named. Here we change one or two things at a time.

In [ ]:
controlled_configs = [
    ("wider_hidden", {**baseline_config, "hidden_dim": 96, "seed": 11}),
    ("smaller_rep", {**baseline_config, "rep_dim": 6, "seed": 12}),
    ("dropout_decay", {**baseline_config, "dropout": 0.15, "weight_decay": 0.001, "seed": 13}),
]

for name, config in controlled_configs:
    run_experiment(name, config)

columns = ["run_id", "name", "hidden_dim", "rep_dim", "lr", "dropout", "weight_decay", "val_accuracy", "test_accuracy", "parameters"]
print_run_table(tracker.table(), columns)
plot_histories(tracker, "val_acc", "Controlled runs: validation accuracy")
plot_final_metrics(tracker)

---

## 6. Coding block 1: add two tracked runs

**Goal:** make two controlled changes, track them, and compare with the existing runs.

Good changes:

- increase or decrease `rep_dim`;
- adjust `lr`;
- add dropout;
- change `batch_size`;
- train for a few more epochs.

Avoid changing everything at once. The point is not magic tuning. The point is evidence.

In [ ]:
# TODO: edit one or two values in each config, then re-run.
student_configs = [
    ("student_low_lr", {**baseline_config, "lr": 0.003, "epochs": 28, "seed": 21}),
    ("student_bigger_rep", {**baseline_config, "rep_dim": 24, "hidden_dim": 80, "seed": 22}),
]

for name, config in student_configs:
    run_experiment(name, config)

print_run_table(tracker.table(), columns)
plot_histories(tracker, "val_acc", "All tracked runs after coding block 1")
plot_final_metrics(tracker)

best_run = tracker.best("val_accuracy")
print("Best validation run:", best_run["run_id"], best_run["name"], best_run["metrics"])

### Run-comparison report prompt

Write 4-6 sentences:

1. Which run had the best validation score?
2. Which run had the best test score?
3. Did a larger model always help?
4. Which run would you choose, and why?
5. What extra information would you want before shipping this model?

---

## 7. Hyperparameter tuning intuition

Optuna and similar tools automate a loop like this:

1. propose a configuration;
2. train/evaluate it;
3. record the result;
4. use previous results to propose better configurations.

Here we implement the smallest possible version: random search. This is less clever than Optuna, but it teaches the same workflow shape.

### Search strategy depends on experiment cost

| Situation | Reasonable strategy | Why |
|---|---|---|
| Tiny model, tiny search space | Full grid search | You can afford to try every combination and compare cleanly. |
| Medium model, many possible settings | Random search or Optuna | You sample the space instead of pretending every combination is affordable. |
| Large model or expensive fine-tuning | Controlled ladder | Change one parameter at a time, keep the best stable setting, then move to the next parameter. |

A controlled ladder is slower intellectually but cheaper computationally: first choose a learning-rate range, then batch size, then regularization, then representation size. Each step asks, "given what we already chose, does this one change help?"

In [ ]:
def sample_config(trial_id, seed=100):
    rng = np.random.default_rng(seed + trial_id)
    hidden_choices = [32, 48, 64, 96, 128]
    rep_choices = [6, 10, 12, 16, 24, 32]
    lr_choices = [0.001, 0.003, 0.006, 0.01, 0.02]
    dropout_choices = [0.0, 0.1, 0.2, 0.3]
    wd_choices = [0.0, 0.0005, 0.001, 0.003]
    return {
        "hidden_dim": int(rng.choice(hidden_choices)),
        "rep_dim": int(rng.choice(rep_choices)),
        "dropout": float(rng.choice(dropout_choices)),
        "lr": float(rng.choice(lr_choices)),
        "weight_decay": float(rng.choice(wd_choices)),
        "batch_size": 128,
        "epochs": 18,
        "seed": int(100 + trial_id),
    }

search_tracker = RunTracker("week10_random_search")

for trial_id in range(6):
    config = sample_config(trial_id)
    run_experiment(f"trial_{trial_id}", config, tracker=search_tracker, verbose=False)

search_columns = ["run_id", "name", "hidden_dim", "rep_dim", "lr", "dropout", "weight_decay", "val_accuracy", "test_accuracy", "parameters"]
print_run_table(search_tracker.table(), search_columns)
plot_histories(search_tracker, "val_acc", "Random-search validation curves")
plot_final_metrics(search_tracker)

best_trial = search_tracker.best("val_accuracy")
print("Best trial by validation accuracy:", best_trial["name"], best_trial["config"], best_trial["metrics"])

In [ ]:
def plot_search_space(search_tracker):
    rows = search_tracker.table()
    params = np.array([row["parameters"] for row in rows])
    rep_dims = np.array([row["rep_dim"] for row in rows])
    val_acc = np.array([row["val_accuracy"] for row in rows])
    names = [row["name"] for row in rows]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(params, val_acc, s=80)
    axes[0].set_xlabel("trainable parameters")
    axes[0].set_ylabel("validation accuracy")
    axes[0].set_title("Size vs validation")
    for name, x, yv in zip(names, params, val_acc):
        axes[0].text(x, yv + 0.003, name, fontsize=8)

    axes[1].scatter(rep_dims, val_acc, s=80, color="#f58518")
    axes[1].set_xlabel("representation dimension")
    axes[1].set_ylabel("validation accuracy")
    axes[1].set_title("Representation size vs validation")
    for name, x, yv in zip(names, rep_dims, val_acc):
        axes[1].text(x, yv + 0.003, name, fontsize=8)

    plt.tight_layout()
    plt.show()

plot_search_space(search_tracker)

### Optional: the same search as an Optuna objective

The course syllabus explicitly names Optuna, so it is useful to see the real shape of the API. This cell is optional because Optuna is not a core dependency for the notebook.

The important mapping is:

- our random-search loop becomes `study.optimize(...)`;
- one proposed configuration becomes a `trial`;
- `trial.suggest_*` defines the search space;
- the objective returns the validation metric to maximize.

In [ ]:
RUN_OPTIONAL_OPTUNA = False  # Set to True after installing optuna.

if not RUN_OPTIONAL_OPTUNA:
    print("Optional Optuna demo skipped. To run it: install optuna, set RUN_OPTIONAL_OPTUNA=True, and re-run this cell.")
else:
    try:
        import optuna
    except ImportError:
        print("Optuna is not installed. In Colab, run: %pip install optuna")
    else:
        def objective(trial):
            config = {
                "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 48, 64, 96, 128]),
                "rep_dim": trial.suggest_categorical("rep_dim", [6, 10, 12, 16, 24, 32]),
                "dropout": trial.suggest_float("dropout", 0.0, 0.3, step=0.1),
                "lr": trial.suggest_float("lr", 1e-3, 2e-2, log=True),
                "weight_decay": trial.suggest_float("weight_decay", 1e-5, 3e-3, log=True),
                "batch_size": 128,
                "epochs": 18,
                "seed": 300 + trial.number,
            }
            optuna_tracker = RunTracker("optional_optuna_single_trial")
            _, run = run_experiment(
                f"optuna_trial_{trial.number}",
                config,
                tracker=optuna_tracker,
                verbose=False,
            )
            trial.set_user_attr("test_accuracy", run["metrics"]["test_accuracy"])
            trial.set_user_attr("parameters", run["metrics"]["parameters"])
            return run["metrics"]["val_accuracy"]

        sampler = optuna.samplers.TPESampler(seed=10)
        study = optuna.create_study(direction="maximize", sampler=sampler)
        study.optimize(objective, n_trials=6)

        print("best validation accuracy:", round(study.best_value, 4))
        print("best params:", study.best_params)
        print("best trial user attrs:", study.best_trial.user_attrs)

### Tuning discussion

A tuning result is not automatically a good final answer.

Ask:

- Did the best validation run also do well on test?
- Was the win large enough to matter?
- Was the run stable, or lucky?
- Is the best model too large or slow for the task?
- What would happen if we changed the random seed?

---

## 8. Practical fine-tuning workflow

Fine-tuning is not one action. It is a decision between options:

| Option | What changes? | When it helps |
|---|---|---|
| Train from scratch | all weights start random | enough labeled data, small model, simple task |
| Freeze representation | keep feature extractor fixed, train only head | little data, useful existing representation |
| Fine-tune | start from pretrained weights, update some or all weights | existing representation is useful but not perfectly matched |

We simulate this with the same digits data. First we train a 10-class source model, then adapt its representation to a smaller even-vs-odd target task.

In [ ]:
source_config = {**baseline_config, "hidden_dim": 96, "rep_dim": 24, "epochs": 28, "seed": 50}
source_tracker = RunTracker("source_pretraining")
source_model, source_run = run_experiment("source_10_class", source_config, tracker=source_tracker)
print_run_table(source_tracker.table(), columns)

# Build a small target task: even vs odd, with very little labeled training data.
y_binary = (y % 2).astype("int64")
X_train_full_b, X_test_b, y_train_full_b, y_test_b = train_test_split(
    X, y_binary, test_size=0.20, random_state=20, stratify=y_binary
)
X_train_b, X_val_b, y_train_b, y_val_b = train_test_split(
    X_train_full_b, y_train_full_b, train_size=120, random_state=20, stratify=y_train_full_b
)

# Use the source-task scaler so copied source representations see the same preprocessing.
X_train_b = scaler.transform(X_train_b).astype("float32")
X_val_b = scaler.transform(X_val_b).astype("float32")
X_test_b = scaler.transform(X_test_b).astype("float32")

binary_splits = {
    "train": (torch.tensor(X_train_b), torch.tensor(y_train_b)),
    "val": (torch.tensor(X_val_b), torch.tensor(y_val_b)),
    "test": (torch.tensor(X_test_b), torch.tensor(y_test_b)),
}

print("binary train / val / test:", len(y_train_b), len(y_val_b), len(y_test_b))

In [ ]:
def make_binary_model_from_source(source_model=None, freeze_features=False, seed=0):
    set_seed(seed)
    model = RepresentationMLP(input_dim=64, hidden_dim=96, rep_dim=24, output_dim=2, dropout=0.0)
    if source_model is not None:
        model.features.load_state_dict(copy.deepcopy(source_model.features.state_dict()))
    if freeze_features:
        for p in model.features.parameters():
            p.requires_grad = False
    return model


def run_binary_experiment(name, model, config, tracker):
    start = time.time()
    model, history = train_model(
        model,
        binary_splits["train"],
        binary_splits["val"],
        epochs=config["epochs"],
        lr=config["lr"],
        weight_decay=config.get("weight_decay", 0.0),
        batch_size=config.get("batch_size", 64),
        seed=config.get("seed", 0),
    )
    val_metrics = evaluate(model, *binary_splits["val"])
    test_metrics = evaluate(model, *binary_splits["test"])
    metrics = {
        "val_accuracy": round(val_metrics["accuracy"], 4),
        "test_accuracy": round(test_metrics["accuracy"], 4),
        "parameters": count_parameters(model),
        "seconds": round(time.time() - start, 2),
    }
    return tracker.log_run(name, config, metrics, history, artifacts={"model": model})

transfer_tracker = RunTracker("binary_transfer_decision")

binary_configs = [
    ("scratch", make_binary_model_from_source(None, False, seed=70), {"strategy": "scratch", "epochs": 30, "lr": 0.01, "batch_size": 64, "seed": 70}),
    ("frozen_head", make_binary_model_from_source(source_model, True, seed=71), {"strategy": "freeze_features", "epochs": 30, "lr": 0.01, "batch_size": 64, "seed": 71}),
    ("fine_tune", make_binary_model_from_source(source_model, False, seed=72), {"strategy": "fine_tune", "epochs": 30, "lr": 0.003, "batch_size": 64, "seed": 72}),
]

for name, model, config in binary_configs:
    run_binary_experiment(name, model, config, transfer_tracker)

print_run_table(transfer_tracker.table(), ["run_id", "name", "strategy", "lr", "val_accuracy", "test_accuracy", "parameters", "seconds"])
plot_histories(transfer_tracker, "val_acc", "Freeze vs fine-tune validation curves")
plot_final_metrics(transfer_tracker)

### Fine-tuning decision prompt

Write 4-6 sentences:

1. Which strategy worked best on validation?
2. Which strategy used the fewest trainable parameters?
3. Did the pretrained representation help?
4. Would you freeze, fine-tune, or train from scratch for a real small-data task?
5. What additional check would you run before trusting the result?

---

## 9. Assignment 5 preview

Assignment 5 will ask you to run, track, compare, and explain training configurations.

Minimum evidence will likely include:

- a baseline run;
- at least four tracked experiments;
- a small hyperparameter search;
- validation/test comparison;
- at least one plot comparing runs;
- a short fine-tuning or transfer-learning decision;
- a reproducibility checklist.

The main question is not "which number is highest?" The main question is: **what evidence supports your modeling decision?**